In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
from scipy.linalg import solve
import string


# --- 1. Model Parameters & LNA ---
def get_model_para():
    mu = 0.03; alpha = 0.15; ga1 = 0.15; ga2 = 0.15
    bt1 = 3.2; bt2 = 2.0; rho1 = 0.5; rho2 = 0.5
    sg1 = 0.3; sg2 = 0.46
    
    # 1. Steady State
    A1 = (mu + ga1) * bt1 * bt2
    A2 = (mu + ga1) * ((mu + alpha) * bt2 + (mu + ga2) * bt1) - bt1 * bt2 * mu
    A3 = (mu + ga1) * (mu + alpha) * (mu + ga2)
    R0 = (bt1 * mu) / ((mu + alpha) * (mu + ga1)) + (bt2 * alpha * mu) / ((mu + ga2) * (mu + alpha) * (mu + ga1))
    
    Istar = (-A2 + np.sqrt(A2**2 - 4 * A1 * A3 * (1 - R0))) / (2 * A1)
    Sstar = mu / (mu + alpha + bt1 * Istar)
    Vstar = (alpha * mu) / ((mu + alpha + bt1 * Istar) * (mu + ga2 + bt2 * Istar))
    mean = np.array([Sstar, Vstar, Istar])
    
    # 2. Linearization Matrices (Reduced for brevity, logic identical to R)
    a11 = mu + alpha + bt1 * Istar; a13 = bt1 * Sstar
    a21 = alpha; a22 = mu + ga2 + bt2 * Istar; a23 = bt2 * Vstar
    a31 = bt1 * Istar; a32 = bt2 * Istar; a34 = bt1 * Sstar * Istar; a35 = bt2 * Vstar * Istar
    
    tao1 = a11 + a22; tao2 = a11 * a22 + a13 * a31 + a23 * a32
    tao3 = a13 * a22 * a31 + a13 * a21 * a32 + a11 * a23 * a32
    
    # Delta 1 & Sig 1
    a1_val = rho1**2*(tao1*tao2-tao3)+tao2*(rho1*tao2+tao3)
    a2_val = rho1*tao2+tao3; a3_val = rho1+tao1
    a4_val = rho1*tao1*(rho1+tao1)+tao1*tao2-tao3
    a5_val = 2*(rho1**3+rho1**2*tao1+rho1*tao2+tao3)*(tao1*tao2-tao3)
    
    delta1 = np.array([
        [a1_val/a5_val, 0, -a2_val/a5_val, 0, 0],
        [0, a2_val/a5_val, 0, -a3_val/a5_val, 0],
        [-a2_val/a5_val, 0, a3_val/a5_val, 0, 0],
        [0, -a3_val/a5_val, 0, a4_val/(rho1*tao3*a5_val), 0],
        [0, 0, 0, 0, 0]
    ])
    
    A = -a13*a22 - a23*a32; B = -a13*a21 - a11*a23 + a23*a31; C = a11*a22 - a22*a31 - a21*a32
    F1 = np.array([
        [0, -1, -a13-a22, A, 0],
        [0, 0, -(a21+a23), B, 0],
        [0, 1, -(a31-a11-a22), C, 0],
        [1/a34, tao1/a34, tao2/a34, tao3/a34, 0],
        [0, 0, 0, 0, 0]
    ])
    Sig1 = (a34**2)*(sg1**2) * F1 @ delta1 @ F1.T
    
    # Delta 2 & Sig 2
    D = -a13*(a32-a22)
    E = a13*(a21+a31+((-a11+a22-a32)*(a22-a23-a32))/a13)+(-a22+a32)*(-a11+a22-a23-a32)
    F_val = a11*(a32-a22)
    F2 = np.array([
        [0, 0, a13, D, 0],
        [0, 1, a11+a23, E, 0],
        [0, -1, a32-a11-a22, F_val, 0],
        [0, 0, 0, 0, 0],
        [-1/a35, -tao1/a35, -tao2/a35, -tao3/a35, 0]
    ])
    
    a1_val = rho2**2*(tao1*tao2-tao3)+tao2*(rho2*tao2+tao3)
    a2_val = rho2*tao2+tao3; a3_val = rho2+tao1
    a4_val = rho2*tao1*(rho2+tao1)+tao1*tao2-tao3
    a5_val = 2*(rho2**3+rho2**2*tao1+rho2*tao2+tao3)*(tao1*tao2-tao3)
    
    delta2 = np.array([
        [a1_val/a5_val, 0, -a2_val/a5_val, 0, 0],
        [0, a2_val/a5_val, 0, -a3_val/a5_val, 0],
        [-a2_val/a5_val, 0, a3_val/a5_val, 0, 0],
        [0, -a3_val/a5_val, 0, a4_val/(rho2*tao3*a5_val), 0],
        [0, 0, 0, 0, 0]
    ])
    Sig2 = (a35**2)*(sg2**2) * F2 @ delta2 @ F2.T
    
    Sigma = (Sig1 + Sig2)[0:3, 0:3]
    return mean, Sigma

# --- 2. SDE Simulation ---
def sde_Euler_traj(n, t):
    h = t / n
    sqrt_h = np.sqrt(h)
    mu = 0.03; alpha = 0.15; ga1 = 0.15; ga2 = 0.15
    bt1 = 3.2; bt2 = 2.0; rho1 = 0.5; rho2 = 0.5; sg1 = 0.3; sg2 = 0.46
    
    S = np.zeros(n); V = np.zeros(n); I = np.zeros(n)
    S_det = np.zeros(n); V_det = np.zeros(n); I_det = np.zeros(n)
    cc1 = np.zeros(n); cc2 = np.zeros(n)
    
    # Init
    S[0], V[0], I[0] = 0.35, 0.25, 0.30
    S_det[0], V_det[0], I_det[0] = 0.35, 0.25, 0.30
    cc1[0] = np.log(bt1); cc2[0] = np.log(bt2)
    
    dW = np.random.normal(0, 1, (n, 2))
    
    for j in range(1, n):
        # OU
        cc1[j] = cc1[j-1] + rho1*(np.log(bt1)-cc1[j-1])*h + sg1*dW[j,0]*sqrt_h
        cc2[j] = cc2[j-1] + rho2*(np.log(bt2)-cc2[j-1])*h + sg2*dW[j,1]*sqrt_h
        beta1_t = np.exp(cc1[j-1])
        beta2_t = np.exp(cc2[j-1])
        
        # Stochastic
        dS = mu - (mu+alpha)*S[j-1] - beta1_t*S[j-1]*I[j-1]
        dV = alpha*S[j-1] - beta2_t*V[j-1]*I[j-1] - (mu+ga2)*V[j-1]
        dI = beta1_t*S[j-1]*I[j-1] + beta2_t*V[j-1]*I[j-1] - (mu+ga1)*I[j-1]
        S[j] = S[j-1] + dS*h; V[j] = V[j-1] + dV*h; I[j] = I[j-1] + dI*h
        if S[j]<0: S[j]=0; 
        if V[j]<0: V[j]=0; 
        if I[j]<0: I[j]=0
            
        # Deterministic
        dS_d = mu - (mu+alpha)*S_det[j-1] - bt1*S_det[j-1]*I_det[j-1]
        dV_d = alpha*S_det[j-1] - bt2*V_det[j-1]*I_det[j-1] - (mu+ga2)*V_det[j-1]
        dI_d = bt1*S_det[j-1]*I_det[j-1] + bt2*V_det[j-1]*I_det[j-1] - (mu+ga1)*I_det[j-1]
        S_det[j] = S_det[j-1] + dS_d*h; V_det[j] = V_det[j-1] + dV_d*h; I_det[j] = I_det[j-1] + dI_d*h
        
    return (S, V, I), (S_det, V_det, I_det)

def run_experiment_3():
    N = int(1e6)
    T_max = 1e4
    
    (S_sto, V_sto, I_sto), (S_det, V_det, I_det) = sde_Euler_traj(N, T_max)
    

    mean, Sigma = get_model_para()
    
    # Burn-in 
    start_idx = int(N * 0.5)
    S_clean = S_sto[start_idx:]
    V_clean = V_sto[start_idx:]
    I_clean = I_sto[start_idx:]
    

    fig, axes = plt.subplots(3, 2, figsize=(10, 8), dpi=300) 
    
    time = np.linspace(0, T_max, N)
    sparse = slice(0, N, 100) 
    
    
    col_det = 'black'
    col_sto = '#4169E1' 
    col_dens = '#FF6B6B'
    
    
    vars_sto = [S_sto, V_sto, I_sto]
    vars_det = [S_det, V_det, I_det]
    vars_clean = [S_clean, V_clean, I_clean]
    
    names = ['S', 'V', 'I']
    y_labels_traj = [r'$S(t)$', r'$V(t)$', r'$I(t)$']
    x_labels_dist = [r'$S$', r'$V$', r'$I$']
    
    
    means = mean
    stds = np.sqrt(np.diag(Sigma))
    
    
    letters = string.ascii_lowercase
    
    
    for i in range(3):
        
        
        # === Trajectories ===
        ax_traj = axes[i, 0]
        
        
        ax_traj.plot(time[sparse], vars_sto[i][sparse], color=col_sto, alpha=0.5, lw=0.5, label='Stochastic')
        ax_traj.plot(time[sparse], vars_det[i][sparse], color=col_det, lw=1.5, label='Deterministic')
        
        
        ax_traj.set_ylabel(y_labels_traj[i], fontsize=12, fontweight='bold')
        if i == 2: 
            ax_traj.set_xlabel('Time', fontsize=12, fontweight='bold')
            
        
        label_char = letters[i * 2] 
        ax_traj.text(-0.15, 1.05, f'({label_char})', transform=ax_traj.transAxes, 
                     fontsize=14, fontweight='bold', va='top', ha='right')
        
        
        if i == 0:
            ax_traj.legend(frameon=False, loc='upper right', fontsize=10)

        # === Distributions ===
        ax_dist = axes[i, 1]
        
        
        ax_dist.hist(vars_clean[i], bins=60, density=True, color='white', edgecolor='grey', alpha=0.8)
        
        
        x_min, x_max = ax_dist.get_xlim()
        x_grid = np.linspace(x_min, x_max, 200)
        pdf = stats.norm.pdf(x_grid, means[i], stds[i])
        
        ax_dist.plot(x_grid, pdf, color=col_dens, lw=2.5, label='LNA PDF')
        ax_dist.fill_between(x_grid, pdf, color=col_dens, alpha=0.3)
        
        ax_dist.set_xlabel(x_labels_dist[i], fontsize=12, fontweight='bold')
        ax_dist.set_ylabel('Density', fontsize=12, fontweight='bold') 
        
        
        label_char = letters[i * 2 + 1] 
        ax_dist.text(-0.15, 1.05, f'({label_char})', transform=ax_dist.transAxes, 
                     fontsize=14, fontweight='bold', va='top', ha='right')
        
        if i == 0:
            ax_dist.legend(frameon=False, loc='upper right', fontsize=10)

    plt.tight_layout()
    
    
    filename = 'Trajectory_and_Distribution.pdf'
    plt.savefig(filename, format='pdf', bbox_inches='tight')
    print(f"Figure saved as {filename}")
    
    plt.show()

if __name__ == "__main__":
    run_experiment_3()